# Database overview

Status of the project database: table sizes, entity classes, imported files,
attribute vocabularies and relations. Adapted from the timelink-py web tests
notebook to run against the project database through `TimelinkNotebook`.

In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info()

## Table row counts

In [ ]:
tlnb.table_row_count_df()

## Entities by class

In [ ]:
import pandas as pd
from sqlalchemy import select, func
from timelink.api.models import Entity

stmt = select(Entity.pom_class, func.count().label('count')).group_by(Entity.pom_class)

with tlnb.db.session() as session:
    pom_class_df = pd.DataFrame(session.execute(stmt), columns=['pom_class', 'count'])
pom_class_df

## Imported files

In [ ]:
imported = tlnb.db.get_imported_files()
imported_df = pd.DataFrame([dict(file) for file in imported])
imported_df

## Attributes vocabulary

Number of attributes and distinct values per type — the place to discover the
attribute types of this project before filtering by value:

In [ ]:
from sqlalchemy import func, select

attr_table = tlnb.db.get_table('attributes')
stmt = select(
    attr_table.c.the_type,
    func.count().label('count'),
    func.count(func.distinct(attr_table.c.the_value)).label('distinct_value'),
).group_by('the_type')

with tlnb.db.session() as session:
    attr_types_df = pd.DataFrame(session.execute(stmt), columns=['the_type', 'count', 'distinct_values'])
attr_types_df.sort_values('count', ascending=False)

## Relations by type

In [ ]:
rel = tlnb.db.get_table('relations')
stmt = select(
    rel.c.the_type,
    func.count().label('count'),
    func.count(func.distinct(rel.c.the_value)).label('distinct_value'),
).group_by('the_type')

with tlnb.db.session() as session:
    rel_types_df = pd.DataFrame(session.execute(stmt), columns=['the_type', 'count', 'distinct_values'])
rel_types_df.sort_values('count', ascending=False)

## Relations with names (view)

The `nrelations` view joins relations with the names of origin and destination
entities; useful to browse a specific relation type:

In [ ]:
nrels = tlnb.db.get_view('nrelations')
tlnb.db.describe(nrels, show=True)

stmt = select(nrels).limit(10)
with tlnb.db.session() as session:
    sample_df = pd.DataFrame(session.execute(stmt))
sample_df